In [1]:
#%pip install pyserial

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


import serial
import threading
import time


### Loading the Model

In [3]:
# Train REGULARIZED version
class EmotionCNN_Reg(nn.Module):
    def __init__(self, input_channels=20, num_classes=4):
        super().__init__()
        self.conv1 = nn.Conv1d(input_channels, 16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm1d(16)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm1d(32)
        self.pool2 = nn.MaxPool1d(2)

        self.fc1       = nn.Linear(32 * 31, 64)
        self.dropout1  = nn.Dropout(0.3)
        self.fc2       = nn.Linear(64, 32)
        self.dropout2  = nn.Dropout(0.5)
        self.fc3       = nn.Linear(32, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool1(x)
        x = F.dropout(x, 0.2)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool2(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

# Retrain with regularization
'''model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],
                           num_classes=4).to(device)
optimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

print("🛡️ Training regularized model...")
print("Input channels:", X_train_torch.shape[1])
print("Total params:", sum(p.numel() for p in model_reg.parameters()))
'''

'model_reg = EmotionCNN_Reg(input_channels=X_train_torch.shape[1],\n                           num_classes=4).to(device)\noptimizer_reg = optim.Adam(model_reg.parameters(), lr=0.001, weight_decay=1e-3)\ncriterion = nn.CrossEntropyLoss()\n\nprint("🛡️ Training regularized model...")\nprint("Input channels:", X_train_torch.shape[1])\nprint("Total params:", sum(p.numel() for p in model_reg.parameters()))\n'

In [4]:
# Load the best CNN model

PATH= 'best_emotion_cnn_reg_125_windowsize.pth'

model = EmotionCNN_Reg()

model.load_state_dict(torch.load(PATH, weights_only=True))
model.eval()


EmotionCNN_Reg(
  (conv1): Conv1d(20, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=992, out_features=64, bias=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc3): Linear(in_features=32, out_features=4, bias=True)
)

### Live Data Streaming

In [ ]:
# Start Data streaming
# How is the live data coming in?

COM_PORT='/dev/cu.usbmodem21301'
# ser = serial.Serial(COM_PORT, 115200) 

window_size = 125
overlap_size = int(0.25 * window_size)
step_size = window_size - overlap_size # 94

live_buffer = []               # The shared "bowl"
buffer_lock = threading.Lock() # The "Pause" button to prevent data crashes

def data_collection_thread():
    # Open the serial port inside the producer
    ser = serial.Serial(COM_PORT, 115200)
    ser.reset_input_buffer() # Clear out old junk!
    
    print("Producer: Listening to Arduino at 30Hz...")
    
    while True:
        try:
            # 1. Catch the data
            raw_line = ser.readline().decode('utf-8').strip()
            sample_list=[]
            for x in raw_line.split(','):
                print("This should be values within the line starting with host time: ", x)
                sample_list.append(float(x))
            #row_data = [float(x) for x in raw_line.split(',')]
            print("Raw Line: ", raw_line) # debug
            #raw_data=row_data[0]
            #print(raw_data)# debug
            #row_data.append(sample_list) # Append the list of values as a single row in the buffer
            
            # 2. Lock the buffer, add the data, unlock
            with buffer_lock:
                live_buffer.append(sample_list)
                
        except Exception as e:
            # If a garbled string comes through, ignore it and keep going
            pass


def inference_thread():
    print("Consumer: Waiting for 125 rows...")
    while True:
        data_to_process = None
        # 1. Safely check the buffer size
        with buffer_lock:
            if len(live_buffer) >= window_size:
                # Copy the first 125 rows for PyTorch
                data_to_process = live_buffer[:window_size]
                
                # The Slide: Delete the oldest 94 rows IN PLACE, leaving the 31 overlap
                del live_buffer[:step_size] 
        
        # 2. Do the heavy ML math OUTSIDE the lock!
        if data_to_process is not None:
            print("Window extracted! Buffer safely sliced.")
            print("Data to Process: ",data_to_process)
        
            # Normalize the data
            data_to_process = np.array(data_to_process)
            mean = np.mean(data_to_process, axis=0)
            std = np.std(data_to_process, axis=0)
            data_to_process = np.subtract(data_to_process, mean) / (std + 1e-8)
            print("Data Normalized", data_to_process.shape)

            with torch.no_grad():
                input_tensor = torch.tensor(data_to_process, dtype=torch.float32)
                output = model(input_tensor)
            print(f"Output: {output}")

            # model(data_to_process)
            # print("Prediction: Stressed!")
            
        else:
            # If we don't have 125 rows yet, sleep for a tiny fraction of a second 
            # so we don't max out the computer's CPU while waiting.
            time.sleep(0.01)




# Create the worker threads
t1 = threading.Thread(target=data_collection_thread, daemon=True)
t2 = threading.Thread(target=inference_thread, daemon=True)

# Start them
t1.start()
t2.start()

# Keep the main script alive so the threads can run in the background
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("Shutting down the system...")




Consumer: Waiting for 125 rows...
Producer: Listening to Arduino at 30Hz...
This should be values within the line starting with host time:  .16
This should be values within the line starting with host time:  0.15
This should be values within the line starting with host time:  -0.11
This should be values within the line starting with host time:  -132.56
This should be values within the line starting with host time:  20.12
This should be values within the line starting with host time:  320.37
This should be values within the line starting with host time:  -0.06
This should be values within the line starting with host time:  0.00
This should be values within the line starting with host time:  0.06
This should be values within the line starting with host time:  -0.02
This should be values within the line starting with host time:  0.00
This should be values within the line starting with host time:  0.00
This should be values within the line starting with host time:  74.81
This should be val

Exception in thread Thread-5 (inference_thread):
Traceback (most recent call last):
  File "/Users/bossbaby07/.pyenv/versions/3.14.2/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/Users/bossbaby07/.pyenv/versions/3.14.2/lib/python3.14/threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/1m/7xl9twrx6n95gvvnzf_m7_7c0000gn/T/ipykernel_52480/1681432799.py", line 63, in inference_thread
    data_to_process = np.array(data_to_process)
ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (125,) + inhomogeneous part.


This should be values within the line starting with host time:  4290421
This should be values within the line starting with host time:  0.06
This should be values within the line starting with host time:  0.03
This should be values within the line starting with host time:  -0.02
This should be values within the line starting with host time:  -132.56
This should be values within the line starting with host time:  20.12
This should be values within the line starting with host time:  320.37
This should be values within the line starting with host time:  -0.06
This should be values within the line starting with host time:  0.31
This should be values within the line starting with host time:  0.00
This should be values within the line starting with host time:  -0.02
This should be values within the line starting with host time:  -0.01
This should be values within the line starting with host time:  0.00
This should be values within the line starting with host time:  74.81
This should be value

### Data Processing

In [ ]:
# Call Data Processing Pipeline

### Model Prediction

In [ ]:
# Live Predictions

